In [14]:
import logging
import logging.handlers
import queue
import sys

## **Phase 1: The Basics (Stop Using print)**

### **The 5 Standard Log Levels**

| Level | Numeric Value | When to Use It |
| --- | --- | --- |
| **`DEBUG`** | 10 | Detailed diagnostic info, mostly for debugging during development. |
| **`INFO`** | 20 | Confirmation that things are working as expected (e.g., "Server started"). |
| **`WARNING`** | 30 | Something unexpected happened, but the app can still run (Default level). |
| **`ERROR`** | 40 | A serious problem; the app failed to perform a specific function. |
| **`CRITICAL`** | 50 | A fatal error; the entire program might crash or stop running. |

- To get started, we use **logging.basicConfig()**. This sets up a global configuration for the root logger.

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

def devide_numbers(a, b):
    logging.debug(f"Devide {a} by {b}")
    
    if b == 0:
        logging.error("Attempted devide by zero")
        return None
    
    logging.info("Division succeful")
    return a / b

In [3]:
devide_numbers(10, 2)

2026-07-06 13:11:05 - INFO - Division succeful


5.0

In [4]:
devide_numbers(3, 0)

2026-07-06 13:11:05 - ERROR - Attempted devide by zero


## **Phase 2: Intermediate (The Big Four Components)**

- Using the root logger globally via basicConfig is fine for quick scripts, but in a real application or package, it causes chaos. If two different packages try to configure the root logger, they will overwrite each other's settings.

- the **Four Pillars** of Python logging:
    1. **Loggers:** The entry point. Your code calls methods on these to emit logs.
    2. **Handlers:** The targets. They take the log message and send it somewhere (the console, a file, an email, or a network socket).
    3. **Formatters:** The stylists. They define the exact layout of the log message string.
    4. **Filters:** The bouncers. They provide fine-grained control over which log records get passed from a logger to a handler.

**Best Practice: The __name__ Pattern**
> Always create a specific logger for each module using **logging.getLogger(__name__)**. This automatically scopes your logs to the file path structure of your project (e.g., project.core.auth).

In [5]:
# Set Loggers
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# Set The Handlers
consol_handler = logging.StreamHandler(sys.stdout)
file_handler = logging.FileHandler("app.log", mode="a")

# Set Level for individual handlers
consol_handler.setLevel(logging.INFO)
file_handler.setLevel(logging.DEBUG)

# Set The Formatter
formatter = logging.Formatter("%(asctime)s | %(name)s | %(levelname)s | %(message)s")

# Add Style format To The Handlers
consol_handler.setFormatter(formatter)
file_handler.setFormatter(formatter)

# Add Handler to the loger
logger.addHandler(consol_handler)
logger.addHandler(file_handler)

# Usage
def devide_number(a, b):
    logger.debug(f"Devide {a} by {b}")

    if b == 0:
        logger.error("Attempted to devide by zero")
        return None
    
    logger.info("Division succeful")
    return a / b

In [6]:
devide_number(10, 3)

2026-07-06 13:11:05,370 | __main__ | INFO | Division succeful


2026-07-06 13:11:05 - INFO - Division succeful


3.3333333333333335

In [7]:
devide_number(4, 0)

2026-07-06 13:11:05,404 | __main__ | ERROR | Attempted to devide by zero


2026-07-06 13:11:05 - ERROR - Attempted to devide by zero


## **Phase 3: Advanced (Production-Ready Patterns)**

In a professional setup, you shouldn't configure loggers programmatically using Python commands like **addHandler**. It spreads configuration all over your codebase. Instead, use a centralized Configuration Dictionary.

### **1. Dictionary Configuration (dictConfig)**

- This is the standard pattern for modern frameworks like Django and FastAPI. It separates your configuration logic cleanly from your application code.

In [8]:
import logging.config

In [9]:
LOGGING_CONFIG = {
    "version": 1,
    "disable_existing_loggers": False,
    "formatters" : {
        "standard" : {
            "format" : "%(asctime)s - [%(levelname)s] - %(name)s => %(message)s"
        },
        "json" : {"time": "%(asctime)s",
                    "level": "%(levelname)",
                    "logger": "%(name)s",
                    "message": "%(message)s"}
    },
    "handlers": {
        "default": {
            "level": "INFO",
            "formatter": "standard",
            "class": "logging.StreamHandler",
            "stream": "ext://sys.stdout"
        },
        "file": {
            "level": "DEBUG",
            "formatter": "json",
            "class": "logging.handlers.RotatingFileHandler",
            "filename": "production.log",
            "maxBytes": 10485760,
            "backupCount": 5,
            "encoding": "utf8"
        }
    },
    "logger": {
        "": {
            "handlers" : ["default", "file"],
            "level": "DEBUG"
        },
        "third_party_library": {
            "handlers": ["default"],
            "level": "WARNING",
            "propagate": "FALSE"
        }
    }
}

logging.config.dictConfig(LOGGING_CONFIG)
logger1 = logging.getLogger(__name__)

In [10]:
def devide_number(a, b):
    logger1.debug(f"Devide {a} by {b}")

    if b == 0:
        logger1.error("Attempted to devide by zero")
        return None
    
    logger1.info("Division succeful")
    return a / b

In [11]:
devide_number(10, 5)

2026-07-06 13:11:05,500 | __main__ | INFO | Division succeful


2026-07-06 13:11:05 - INFO - Division succeful


2.0

In [12]:
devide_number(10, 0)

2026-07-06 13:11:05,530 | __main__ | ERROR | Attempted to devide by zero


2026-07-06 13:11:05 - ERROR - Attempted to devide by zero


>**Pro Tip on Rotation:** Notice the RotatingFileHandler. In production, never use a basic FileHandler without rotation, or your server will eventually run out of disk space. RotatingFileHandler automatically rolls over to a new file when it hits a size limit (maxBytes).

### **2. Contextual Logging (Adding Trace IDs)**

- When building microservices or asynchronous pipelines, you need a way to track a single request across multiple functions or files. You can use a **LoggerAdapter** to inject contextual data (like a user ID or a transaction UUID) into every log statement automatically.

In [13]:
base_logger = logging.getLogger("request_process")
base_logger.setLevel(logging.INFO)

handler = logging.StreamHandler()
handler.setFormatter("[Trace: %(trace_id)s] => %(message)s")
base_logger.addHandler(handler)

context_data = {"trace_id": "req-99ab-1234"}
adapter = logging.LoggerAdapter(base_logger, context_data)

adapter.info("Fetching user profile data.")
adapter.warning("Database connection latency is high.")

[Trace: %(trace_id)s] => %(message)s
2026-07-06 13:11:05 - INFO - Fetching user profile data.
[Trace: %(trace_id)s] => %(message)s
2026-07-06 13:11:05 - WARNING - Database connection latency is high.


## **Phase 4: Expert (Performance Optimization)**

- When you are logging thousands of entries per second, standard logging can block your application threads because writing to disk or stdout is an I/O-bound bottleneck.

### **Non-blocking / Asynchronous Logging**

- To fix this, you can push log operations to a background thread using a **QueueHandler** and a **QueueListener**. The main application thread simply drops the log message into an in-memory queue and instantly returns to processing code.

In [15]:
target_handler = logging.StreamHandler()
formatter = logging.Formatter("%(asctime)s - %(message)s")
target_handler.setFormatter(formatter)

log_queue = queue.Queue(-1)
queue_listener = logging.handlers.QueueListener(log_queue, target_handler)
queue_listener.start()

root_logger = logging.getLogger(__name__)
root_logger.setLevel(logging.INFO)

queue_handler = logging.handlers.QueueHandler(log_queue)
root_logger.addHandler(queue_handler)

root_logger.info("This log statement is entirely non-blocking!")

queue_listener.stop()

2026-07-06 13:41:29,718 | __main__ | INFO | This log statement is entirely non-blocking!


2026-07-06 13:41:29 - INFO - This log statement is entirely non-blocking!
2026-07-06 13:41:29,718 - This log statement is entirely non-blocking!


- **Cheat Sheet: Golden Rules for Production Logging**
    1. Never use logging.root directly. Use **logging.getLogger(__name__)**.
    2. Use the right levels. Don't dump everything into INFO or ERROR.
    3. Never log secrets. Be hyper-vigilant about filtering out passwords, API tokens, and personally identifiable information (PII).
    4. Use **logger.exception()** when handling exceptions to preserve your tracebacks.
    5. Set **propagate = False** on noisy third-party loggers if you don't want them polluting your root handler outputs.